In [2]:
pip install numpy pandas scipy scikit-learn torch joblib

In [3]:
from pathlib import Path
import os, random, zipfile, shutil, json, warnings
from collections import Counter
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# When running in Colab, either upload the ZIP or set ZIP_PATH to the mounted
# Drive location, e.g. '/content/drive/MyDrive/Early Fusion Model.zip'.
ZIP_PATH = os.environ.get('TNAM_ZIP_PATH', '')
if not ZIP_PATH:
    if 'google.colab' in __import__('sys').modules:
        from google.colab import files
        uploaded = files.upload()  # select Early Fusion Model -....zip
        candidates = [n for n in uploaded if n.lower().endswith('.zip')]
        if len(candidates) != 1:
            raise ValueError('Upload exactly one ZIP file.')
        ZIP_PATH = candidates[0]
    else:
        candidates = list(Path('/mnt/data').glob('Early Fusion Model*.zip'))
        if not candidates:
            candidates = list(Path('.').glob('*.zip'))
        if len(candidates) != 1:
            raise ValueError('Set TNAM_ZIP_PATH to your uploaded ZIP path.')
        ZIP_PATH = str(candidates[0])

ZIP_PATH = Path(ZIP_PATH).expanduser()
assert ZIP_PATH.is_file(), f'ZIP not found: {ZIP_PATH}'
WORKDIR = Path(os.environ.get('TNAM_WORKDIR', '/content/tnam_early_fusion'
                               if 'google.colab' in __import__('sys').modules
                               else '/mnt/data/tnam_early_fusion'))
INPUT_DIR = WORKDIR / 'input'
OUTPUT_DIR = WORKDIR / 'results'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ZIP members from this upload have benign spaces; check for traversal.
with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in archive.namelist():
        target = (INPUT_DIR / member).resolve()
        if not target.is_relative_to(INPUT_DIR.resolve()):
            raise ValueError('Unsafe ZIP member path')
    archive.extractall(INPUT_DIR)

# Index by filename/relative path, preserving full folder structure.
def find_input(filename, split_group=None):
    candidates = list(INPUT_DIR.rglob(filename))
    if split_group:
        candidates = [p for p in candidates if split_group in p.parts]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename} in '
                                f'{split_group or "ZIP root"}, found {candidates}')
    return candidates[0]

print('Using ZIP:', ZIP_PATH.name)
for filename in ['expression_aligned.csv', 'mutation_aligned.csv',
                 'cnv_aligned.csv', 'proteomics_aligned.csv',
                 'sex_labels_matched.csv']:
    print('Found', find_input(filename).relative_to(INPUT_DIR))

Saving Early Fusion Model -20261115T083638Z-1-001.zip to Early Fusion Model -20261115T083638Z-1-001.zip
Using ZIP: Early Fusion Model -20261115T083638Z-1-001.zip
Found Early Fusion Model /expression_aligned.csv
Found Early Fusion Model /mutation_aligned.csv
Found Early Fusion Model /cnv_aligned.csv
Found Early Fusion Model /proteomics_aligned.csv
Found Early Fusion Model /sex_labels_matched.csv


In [4]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from scipy.stats import pearsonr, spearmanr
import joblib

RUN_SPLITS = ['unseen_cell_lines', 'unseen_tissues']
N_COMPONENTS = 32              # per omics layer, from training-only SVD
MAX_EPOCHS = int(os.environ.get('TNAM_EPOCHS', 35))
PATIENCE = int(os.environ.get('TNAM_PATIENCE', 6))
BATCH_SIZE = 2048
LR = 1e-3
WEIGHT_DECAY = 1e-4
REWEIGHT_SEX_TISSUE = True      # only for sex-aware variants

SEX_MARKER_GENES = ['XIST', 'RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, os.cpu_count() or 2))
print('PyTorch device:', DEVICE)
print('Training epochs:', MAX_EPOCHS, '(early stopping may finish earlier)')

PyTorch device: cpu
Training epochs: 35 (early stopping may finish earlier)


In [5]:
OMICS_FILENAMES = {
    'expression': 'expression_aligned.csv',
    'mutation': 'mutation_aligned.csv',
    'cnv': 'cnv_aligned.csv',
    'proteomics': 'proteomics_aligned.csv',
}
omics = {}
for layer, fname in OMICS_FILENAMES.items():
    df = pd.read_csv(find_input(fname), low_memory=False)
    assert 'SIDM' in df.columns, f'{fname} missing SIDM'
    if df.SIDM.duplicated().any():
        raise ValueError(f'{fname}: duplicate SIDM IDs')
    df = df.set_index('SIDM')
    df.index = df.index.astype(str)
    omics[layer] = df.astype(np.float32)
    print(f'{layer:12s}: {df.shape[0]} cell lines x {df.shape[1]} features')

shared_omics_ids = set.intersection(*(set(df.index) for df in omics.values()))
print('IDs appearing in all four omics:', len(shared_omics_ids))

labels = pd.read_csv(find_input('sex_labels_matched.csv'))
required_labels = {'SIDM', 'final_sex', 'usable_sex',
                   'exclude_from_sex_analysis'}
assert required_labels.issubset(labels.columns)
labels['SIDM'] = labels['SIDM'].astype(str)
assert labels['SIDM'].is_unique
labels = labels[['SIDM', 'final_sex', 'usable_sex',
                 'exclude_from_sex_analysis', 'label_confidence']].rename(
                     columns={'SIDM': 'cell_line_id'})


def prepare_responses(split_family):
    frames = []
    for part in ['train', 'val', 'test']:
        p = find_input(f'{part}.csv', split_family)
        df = pd.read_csv(p)
        assert set(['sample_id','cell_line_id','drug_id','tissue',
                    'sex','response','split']).issubset(df.columns)
        assert set(df['split'].astype(str)) == {part}, (split_family, part)
        frames.append(df)
    raw = pd.concat(frames, ignore_index=True)
    raw['cell_line_id'] = raw['cell_line_id'].astype(str)
    raw['drug_id'] = raw['drug_id'].astype(str)
    raw['response'] = pd.to_numeric(raw['response'], errors='coerce')

    if raw['sample_id'].duplicated().any():
        raise ValueError('sample_id duplicates across frozen splits!')
    raw = raw.merge(labels, on='cell_line_id', how='left', validate='many_to_one')

    original_cell_line_sex = raw[['cell_line_id','sex','final_sex']].drop_duplicates()
    sex_conflicts = original_cell_line_sex[
        original_cell_line_sex['final_sex'].isin(['Male','Female']) &
        (original_cell_line_sex['sex'] != original_cell_line_sex['final_sex'])
    ]
    usable = (raw['usable_sex'].eq(True) &
              raw['exclude_from_sex_analysis'].eq(False) &
              raw['final_sex'].isin(['Male','Female']) &
              raw['response'].notna() &
              raw['cell_line_id'].isin(shared_omics_ids))
    qc = raw.loc[~usable, ['cell_line_id', 'sex', 'final_sex',
                            'usable_sex','exclude_from_sex_analysis']].drop_duplicates()
    raw['split_sex_original'] = raw['sex']
    data = raw.loc[usable].copy()
    data['sex'] = data['final_sex']
    data.reset_index(drop=True, inplace=True)

    # Ensure frozen split's held-out dimension remains fully separated AFTER QC.
    train = data.loc[data.split=='train']
    val = data.loc[data.split=='val']
    test = data.loc[data.split=='test']
    if split_family == 'unseen_cell_lines':
        field = 'cell_line_id'
    elif split_family == 'unseen_tissues':
        field = 'tissue'
    elif split_family == 'unseen_drugs':
        field = 'drug_id'
    else:
        raise ValueError('Unknown split family')
    group_sets = [set(d[field]) for d in [train,val,test]]
    if any(group_sets[i] & group_sets[j] for i,j in [(0,1),(0,2),(1,2)]):
        raise ValueError(f'Leakage: {field} overlaps frozen partitions')
    if data['sample_id'].duplicated().any():
        raise ValueError('Duplicate drug-response observations')
    if any(len(d)==0 for d in [train,val,test]):
        raise ValueError('At least one split is empty after QC')
    print(f'\n{split_family}: {len(raw):,} raw rows -> {len(data):,} QC-usable rows')
    print(f'  Unique QC-excluded cell lines: {qc.cell_line_id.nunique()}')
    print(f'  Original sex vs final sex conflicts (retained when usable): '
          f'{len(sex_conflicts)} cell lines')
    print('  Rows per partition:', data.split.value_counts().to_dict())
    print('  Unique lines:', data.cell_line_id.nunique(),
          'drugs:', data.drug_id.nunique())
    print('  Sex counts:', data.groupby(['split','sex']).size().to_dict())
    qc.to_csv(OUTPUT_DIR/f'{split_family}_excluded_sex_metadata.csv', index=False)
    sex_conflicts.to_csv(OUTPUT_DIR/f'{split_family}_sex_label_conflicts.csv', index=False)
    return data

expression  : 526 cell lines x 18025 features
mutation    : 526 cell lines x 1667 features
cnv         : 526 cell lines x 23264 features
proteomics  : 526 cell lines x 8498 features
IDs appearing in all four omics: 526


In [7]:
def make_features(data):
    train = data.split.eq('train').to_numpy()
    cell_ids = pd.Index(data['cell_line_id'].unique(), name='cell_line_id')
    train_cell_ids = pd.Index(data.loc[train, 'cell_line_id'].unique())
    row_indices = cell_ids.get_indexer(data['cell_line_id'])
    assert (row_indices >= 0).all()

    layer_embeddings = {}
    transforms = {}
    for layer, table in omics.items():
        # Explicit X/Y marker genes belong only to the sex-aware module,
        # not to the base feature representation for either comparison.
        base = table.drop(columns=SEX_MARKER_GENES, errors='ignore')
        x_train = base.loc[train_cell_ids].to_numpy(dtype=np.float32)
        x_all = base.loc[cell_ids].to_numpy(dtype=np.float32)
        imputer = SimpleImputer(strategy='median', keep_empty_features=True)
        x_train = imputer.fit_transform(x_train).astype(np.float32)
        x_all = imputer.transform(x_all).astype(np.float32)
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train).astype(np.float32)
        x_all = scaler.transform(x_all).astype(np.float32)
        k = min(N_COMPONENTS, len(train_cell_ids)-1, x_train.shape[1]-1)
        if k < 1:
            raise ValueError('Not enough training cells/features for SVD')
        svd = TruncatedSVD(n_components=k, random_state=SEED, n_iter=4)
        x_train = svd.fit_transform(x_train).astype(np.float32)
        x_all = svd.transform(x_all).astype(np.float32)
        svd_scaler = StandardScaler()
        svd_scaler.fit(x_train)
        layer_embeddings[layer] = svd_scaler.transform(x_all).astype(np.float32)
        transforms[layer] = {'imputer':imputer,'scaler':scaler,
                             'svd':svd,'svd_scaler':svd_scaler,
                             'input_feature_names':base.columns.tolist()}
        print(f'  {layer}: {base.shape[1]} features -> {k} dimensions')

    # Drug identity is suitable for unseen CELL LINES / TISSUES, NOT unseen drugs.
    observed_train_drugs = set(data.loc[train,'drug_id'])
    unknown_drugs = set(data['drug_id']) - observed_train_drugs
    if unknown_drugs:
        raise ValueError(f'{len(unknown_drugs)} drugs absent from training. '
                         'One-hot drug ID is invalid for unseen drugs. '
                         'Provide drug molecular descriptors for this split.')
    try:
        encoder = OneHotEncoder(sparse_output=False,handle_unknown='error',
                                dtype=np.float32)
    except TypeError:  # older sklearn
        encoder = OneHotEncoder(sparse=False,handle_unknown='error',
                                dtype=np.float32)
    encoder.fit(data.loc[train,['drug_id']])
    drug_ohe = encoder.transform(data[['drug_id']]).astype(np.float32)

    expression = omics['expression']
    missing_marker_genes = [g for g in SEX_MARKER_GENES if g not in expression]
    if missing_marker_genes:
        raise ValueError(f'Missing expression sex marker genes: {missing_marker_genes}')
    module = expression.loc[cell_ids, SEX_MARKER_GENES].to_numpy(dtype=np.float32)
    marker_imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    marker_train = expression.loc[train_cell_ids, SEX_MARKER_GENES].to_numpy(dtype=np.float32)
    marker_imputer.fit(marker_train)
    marker_scaler = StandardScaler()
    marker_scaler.fit(marker_imputer.transform(marker_train))
    module = marker_scaler.transform(marker_imputer.transform(module)).astype(np.float32)
    sex_binary = (data['sex']=='Male').to_numpy(dtype=np.float32).reshape(-1,1)
    sex_features = np.column_stack([sex_binary, module[row_indices]]).astype(np.float32)

    variants = {}
    for key, use_protein, use_sex in [
        ('EF1',False,False), ('EF2',True,False),
        ('EF3',False,True), ('EF4',True,True)
    ]:
        layers = ['expression','mutation','cnv']
        if use_protein:
            layers.append('proteomics')
        parts = [layer_embeddings[layer][row_indices] for layer in layers]
        parts.append(drug_ohe)
        if use_sex:
            parts.append(sex_features)
        variants[key] = np.hstack(parts).astype(np.float32)
    transforms['drug_encoder'] = encoder
    transforms['marker_imputer'] = marker_imputer
    transforms['marker_scaler'] = marker_scaler
    transforms['sex_marker_genes'] = list(SEX_MARKER_GENES)
    return variants, transforms


def make_train_weights(data):
    tr = data.loc[data.split.eq('train')]
    counts = tr.groupby(['sex','tissue']).size()
    cell_weights = (len(tr) / (len(counts) * counts))**0.5
    cell_weights = cell_weights.clip(lower=0.5, upper=3.0)
    pairs = pd.MultiIndex.from_frame(tr[['sex','tissue']])
    w = pd.Series(pairs.map(cell_weights).to_numpy(dtype=np.float32))
    return (w.to_numpy() / w.mean()).astype(np.float32)

In [8]:
class EarlyFusionMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def batch_predict(model, x, batch_size=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            batch = torch.from_numpy(x[i:i+batch_size]).to(DEVICE)
            out.append(model(batch).cpu().numpy())
    return np.concatenate(out)


def fit_model(X, data, sex_aware, model_name, family):
    train_mask = data.split.eq('train').to_numpy()
    val_mask = data.split.eq('val').to_numpy()
    test_mask = data.split.eq('test').to_numpy()
    y = data.response.to_numpy(dtype=np.float32)
    Xtr, Xval, Xtest = X[train_mask], X[val_mask], X[test_mask]
    ytr, yval, ytest = y[train_mask], y[val_mask], y[test_mask]

    if sex_aware and REWEIGHT_SEX_TISSUE:
        weights = make_train_weights(data)
    else:
        weights = np.ones(len(ytr), dtype=np.float32)
    train_ds = TensorDataset(torch.from_numpy(Xtr),
                             torch.from_numpy(ytr),
                             torch.from_numpy(weights))
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = EarlyFusionMLP(X.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                                   weight_decay=WEIGHT_DECAY)
    best_loss = float('inf')
    best_state = None
    best_epoch = 0
    no_improve = 0
    history = []

    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        weighted_err_sum = 0.
        batch_examples = 0
        for xb, yb, wb in loader:
            xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = (wb * ((pred-yb)**2)).mean()
            loss.backward()
            optimizer.step()
            weighted_err_sum += float(loss.item())*len(yb)
            batch_examples += len(yb)
        val_pred = batch_predict(model, Xval)
        val_rmse = float(np.sqrt(np.mean((val_pred-yval)**2)))
        train_loss = weighted_err_sum / max(1,batch_examples)
        history.append({'epoch':epoch,'training_weighted_mse':train_loss,
                        'val_rmse':val_rmse})
        if val_rmse < best_loss-1e-5:
            best_loss = val_rmse
            best_epoch = epoch
            best_state = {k:v.detach().cpu().clone()
                          for k,v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if epoch == 1 or epoch % 5 == 0 or no_improve == PATIENCE:
            print(f'    {family}/{model_name} epoch {epoch:02d} '
                  f'train MSE={train_loss:.4f} val RMSE={val_rmse:.4f}')
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    test_pred = batch_predict(model, Xtest)
    return (model, best_epoch, best_loss,
            pd.DataFrame(history), test_pred, test_mask)


def metrics(y, pred):
    if len(y)==0:
        return {'n':0, 'rmse':np.nan, 'pearson':np.nan, 'spearman':np.nan}
    rmse = float(np.sqrt(np.mean((y-pred)**2)))
    if len(y)<3 or np.std(y)==0 or np.std(pred)==0:
        pearson = spearman = np.nan
    else:
        pearson = float(pearsonr(y, pred).statistic)
        spearman = float(spearmanr(y, pred).statistic)
    return {'n':len(y),'rmse':rmse,'pearson':pearson,'spearman':spearman}

In [9]:
EXPERIMENTS = [
    ('EF1', False, False),
    ('EF2', True, False),
    ('EF3', False, True),
    ('EF4', True, True),
]
all_benchmarks = []
all_predictions = []

for family in RUN_SPLITS:
    if family == 'unseen_drugs':
        raise ValueError('Unseen-drug training requires drug molecular '
                         'descriptors or fingerprints not in this ZIP.')
    data = prepare_responses(family)
    print(f'\nPreprocessing features for {family}...')
    variants, fitted_transforms = make_features(data)
    joblib.dump(fitted_transforms, OUTPUT_DIR/f'{family}_preprocessing.joblib',
                compress=3)

    for name, with_protein, sex_aware in EXPERIMENTS:
        print(f'\nTraining {name}: proteomics={with_protein}, sex-aware={sex_aware}; '
              f'X={variants[name].shape}')
        model, epoch, best_val, hist, y_pred, test_mask = fit_model(
            variants[name], data, sex_aware, name, family)
        torch.save({
            'state_dict':model.state_dict(),
            'n_features':variants[name].shape[1],
            'fusion':'early',
            'use_proteomics':with_protein,
            'sex_aware':sex_aware,
            'sex_tissue_reweighting':bool(sex_aware and REWEIGHT_SEX_TISSUE),
            'seed':SEED,
            'best_epoch':epoch,
            'best_val_rmse':best_val,
        }, OUTPUT_DIR/f'{family}_{name}_model.pt')
        hist.to_csv(OUTPUT_DIR/f'{family}_{name}_history.csv',index=False)

        test = data.loc[test_mask].copy()
        y_true = test.response.to_numpy(dtype=np.float32)
        female_mask = test.sex.eq('Female').to_numpy()
        male_mask = test.sex.eq('Male').to_numpy()
        fm = metrics(y_true[female_mask], y_pred[female_mask])
        mm = metrics(y_true[male_mask], y_pred[male_mask])
        om = metrics(y_true, y_pred)
        row = {
            'fusion':'early','split_family':family,
            'model':name,'proteomics':with_protein,
            'sex_aware':sex_aware,
            'n_train':int(data.split.eq('train').sum()),
            'n_val':int(data.split.eq('val').sum()),
            'n_test':len(y_true),'best_epoch':epoch,
            'val_rmse':best_val,
            'overall_rmse':om['rmse'],
            'female_n':fm['n'],'male_n':mm['n'],
            'female_rmse':fm['rmse'],'male_rmse':mm['rmse'],
            'rmse_gap_abs':abs(fm['rmse']-mm['rmse']),
            'rmse_gap_female_minus_male':fm['rmse']-mm['rmse'],
            'female_pearson':fm['pearson'],'male_pearson':mm['pearson'],
            'pearson_gap_abs':abs(fm['pearson']-mm['pearson']),
            'female_spearman':fm['spearman'],'male_spearman':mm['spearman'],
            'spearman_gap_abs':abs(fm['spearman']-mm['spearman']),
        }
        all_benchmarks.append(row)
        print(f'  TEST: overall RMSE={om["rmse"]:.4f} '
              f'Female RMSE={fm["rmse"]:.4f} '
              f'Male RMSE={mm["rmse"]:.4f} '
              f'Gap={row["rmse_gap_abs"]:.4f}')
        test['prediction'] = y_pred
        test['model'] = name
        test['fusion'] = 'early'
        test['split_family'] = family
        all_predictions.append(test[['sample_id','cell_line_id','drug_id',
                                     'tissue','sex','response','prediction',
                                     'split','fusion','model','split_family']])


unseen_cell_lines: 125,360 raw rows -> 119,789 QC-usable rows
  Unique QC-excluded cell lines: 23
  Original sex vs final sex conflicts (retained when usable): 5 cell lines
  Rows per partition: {'train': 83801, 'val': 18092, 'test': 17896}
  Unique lines: 469 drugs: 295
  Sex counts: {('test', 'Female'): 9196, ('test', 'Male'): 8700, ('train', 'Female'): 37781, ('train', 'Male'): 46020, ('val', 'Female'): 8774, ('val', 'Male'): 9318}

Preprocessing features for unseen_cell_lines...
  expression: 18020 features -> 32 dimensions
  mutation: 1667 features -> 32 dimensions
  cnv: 23259 features -> 32 dimensions
  proteomics: 8498 features -> 32 dimensions

Training EF1: proteomics=False, sex-aware=False; X=(119789, 391)
    unseen_cell_lines/EF1 epoch 01 train MSE=10.6008 val RMSE=2.6325
    unseen_cell_lines/EF1 epoch 05 train MSE=2.4936 val RMSE=1.7275
    unseen_cell_lines/EF1 epoch 10 train MSE=1.4118 val RMSE=1.4617
    unseen_cell_lines/EF1 epoch 15 train MSE=1.2789 val RMSE=1.4699

In [10]:
benchmark = pd.DataFrame(all_benchmarks)
benchmark_path = OUTPUT_DIR/'early_fusion_master_benchmark.csv'
benchmark.to_csv(benchmark_path,index=False)

predictions = pd.concat(all_predictions,ignore_index=True)
predictions.to_csv(OUTPUT_DIR/'early_fusion_test_predictions.csv',index=False)

long_rows = []
for _, rec in benchmark.iterrows():
    for group in ['female','male']:
        long_rows.append({'fusion':'early','split_family':rec['split_family'],
                          'model':rec['model'],
                          'proteomics':rec['proteomics'],
                          'sex_aware':rec['sex_aware'],
                          'group':group.title(),
                          'n':rec[f'{group}_n'],
                          'RMSE':rec[f'{group}_rmse'],
                          'Pearson':rec[f'{group}_pearson'],
                          'Spearman':rec[f'{group}_spearman']})
    long_rows.append({'fusion':'early','split_family':rec['split_family'],
                      'model':rec['model'],
                      'proteomics':rec['proteomics'],
                      'sex_aware':rec['sex_aware'],
                      'group':'Gap (absolute)',
                      'n':np.nan,'RMSE':rec['rmse_gap_abs'],
                      'Pearson':rec['pearson_gap_abs'],
                      'Spearman':rec['spearman_gap_abs']})
pd.DataFrame(long_rows).to_csv(
    OUTPUT_DIR/'early_fusion_benchmark_female_male_gap.csv',index=False)

print('\nFinal TEST benchmark (lower RMSE is better):')
print(benchmark[['split_family','model','proteomics','sex_aware',
                 'female_n','male_n','female_rmse','male_rmse',
                 'rmse_gap_abs','overall_rmse','best_epoch']].round(4).to_string(index=False))
print('\nSaved outputs under:',OUTPUT_DIR)
archive_path = shutil.make_archive(str(WORKDIR/'early_fusion_results'),
                                  'zip', root_dir=OUTPUT_DIR)
print('Results ZIP:',archive_path)

# Colab automatically downloads the result archive when training finishes.
if 'google.colab' in __import__('sys').modules:
    from google.colab import files
    files.download(archive_path)


Final TEST benchmark (lower RMSE is better):
     split_family model  proteomics  sex_aware  female_n  male_n  female_rmse  male_rmse  rmse_gap_abs  overall_rmse  best_epoch
unseen_cell_lines   EF1       False      False      9196    8700       1.5967     1.6822        0.0855        1.6388          10
unseen_cell_lines   EF2        True      False      9196    8700       1.6901     1.8402        0.1501        1.7647          12
unseen_cell_lines   EF3       False       True      9196    8700       1.6564     1.6838        0.0274        1.6698          11
unseen_cell_lines   EF4        True       True      9196    8700       1.8357     2.0203        0.1846        1.9276          15
   unseen_tissues   EF1       False      False      7701    8463       2.4160     1.9052        0.5108        2.1637          10
   unseen_tissues   EF2        True      False      7701    8463       2.6130     2.0437        0.5693        2.3323          13
   unseen_tissues   EF3       False       True     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>